<a href="https://colab.research.google.com/github/LucasMartinscode/LucasMartinscode/blob/LucasMartinscode-patch-1/Atividade_09.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# docker file para instalação de faker e kafka dentro do airflow



In [ ]:
FROM apache/airflow:2.5.1-python3.8

RUN pip install 'apache-airflow[amazon]' boto3
RUN pip install faker
RUN pip pip install faker kafka-python




# Arquivo para cronjob no airflow


In [ ]:
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime, timedelta
import pandas as pd
from faker import Faker
import random
import os
import time
from json import dumps, loads
from kafka import KafkaProducer, KafkaConsumer

# Função para gerar dados de Municípios e Estados
def gerar_dados():
    fake = Faker("pt_BR")
    lista_de_usuarios = []
    for _ in range(random.randint(1, 10)):  # Gerando entre 1 e 10 registros por bloco
        data = {
            'codigo_uf': fake.unique.random_int(min=1, max=99999999),
            'uf': fake.state_abbr(),
            'nome': fake.name(),
            'latitude': fake.latitude(),
            'longitude': fake.longitude(),
            'regiao': fake.state()
        }
        lista_de_usuarios.append(data)
    return lista_de_usuarios

# Função para produzir dados para Kafka
def enviar_para_kafka():
    fake = Faker("pt_BR")
    producer = KafkaProducer(bootstrap_servers=['awari-kafka:9093'],
                             value_serializer=lambda x: dumps(x).encode('utf-8'))

    for e in range(10):  # Gerando 10 blocos de dados para teste
        dados = gerar_dados()
        producer.send('municipios', value=dados)
        producer.flush()  # Assegura que os dados foram enviados
        print(f'Dados enviados para o Kafka: {dados}')
        time.sleep(random.randint(1, 2))  # Dorme por 1 a 2 segundos

# Função para consumir dados do Kafka e salvar em CSV
def consumir_de_kafka():
    output_file = r'C:\Users\LucasMartins\OneDrive - SUPERCAMPO S.A\Desktop\Eng de dados\awari-engenharia-de-dados-docker-main\awari-engenharia-de-dados-docker-main\exercicios\municipios-estados\csv\estados.csv'

    if not os.path.exists(os.path.dirname(output_file)):
        os.makedirs(os.path.dirname(output_file))

    # Verifica se o arquivo já existe
    file_exists = os.path.isfile(output_file)

    consumer = KafkaConsumer(
        'municipios',
        bootstrap_servers=['awari-kafka:9093'],
        auto_offset_reset='earliest',
        enable_auto_commit=True,
        group_id='my-group',
        value_deserializer=lambda x: loads(x.decode('utf-8'))
    )

    for message in consumer:
        lista_de_usuarios = message.value
        df = pd.DataFrame(lista_de_usuarios)
        if file_exists:
            df.to_csv(output_file, index=False, header=False, mode='a')  # Adiciona sem cabeçalho
        else:
            df.to_csv(output_file, index=False, header=True, mode='a')  # Adiciona com cabeçalho na primeira vez
            file_exists = True  # Atualiza o estado para não incluir cabeçalho nas próximas vezes
        print(f'Dados adicionados ao arquivo: {output_file}')

default_args = {
    'owner': 'airflow',
    'depends_on_past': False,
    'start_date': datetime(2023, 7, 15),
    'email_on_failure': False,
    'email_on_retry': False,
    'retries': 1,
    'retry_delay': timedelta(minutes=5),
}

dag = DAG(
    'kafka_streaming_dag',
    default_args=default_args,
    description='Um DAG para gerar, enviar e consumir dados via Kafka',
    schedule_interval=timedelta(days=1),
)

t1 = PythonOperator(
    task_id='enviar_para_kafka',
    python_callable=enviar_para_kafka,
    dag=dag,
)

t2 = PythonOperator(
    task_id='consumir_de_kafka',
    python_callable=consumir_de_kafka,
    dag=dag,
)

t1 >> t2
